# 07 — Ensayo del día de la evaluación

El §5 del enunciado describe lo que va a pasar el día 24: se clona el
repositorio, se pone una clave, se pasan **diez preguntas que nadie ha
visto** y hay **veinte minutos**. Este notebook es el ensayo de esa
situación, y su propósito no es sacar buena nota sino descubrir ahora lo que
se rompería entonces.

## Por qué hace falta un hold-out, y por qué el nuestro es imperfecto

Las 40 preguntas de los notebooks 04 y 05 se han mirado muchas veces. Cada
vez que se revisa un fallo y se cambia algo, una parte de la información del
conjunto se filtra a las decisiones de diseño. No es hacer trampa —no hay
ninguna regla escrita para una pregunta concreta— pero sí significa que el
acierto medido sobre ellas es **optimista** como predicción de lo que pasará
con preguntas nuevas.

La forma limpia de estimar eso es un conjunto que no se haya usado para
iterar. El nuestro lo cumple a medias, y conviene decir en qué:

- **Sí lo cumple** en que estas diez preguntas se escriben ahora, de una sola
  pasada, sobre secciones y conceptos que no aparecen en ningún otro golden
  set, y se evalúan **una vez**. No se mira el resultado para cambiar nada:
  si el sistema falla aquí, el fallo se documenta, no se arregla.
- **No lo cumple** en que las hemos escrito nosotros, con el mismo criterio
  con el que escribimos las otras veinte. Un hold-out de verdad lo escribe
  otra persona. El golden set oficial es lo más parecido a eso que tenemos, y
  por eso sus cifras son las que se citan en el informe cuando hace falta una
  medida sin sesgo de autoría.

## Qué se mide aquí que no se mide en el notebook 05

Tres cosas, y ninguna es el acierto:

1. **El tiempo de pared de punta a punta.** No la latencia media por
   pregunta, sino cuánto tarda el proceso entero desde que se llama a
   `evaluar()`. Es lo que tiene que caber en veinte minutos.
2. **Que `evaluar()` funcione sin abrir un notebook**, que es como se va a
   usar. Se comprueba también desde la línea de órdenes.
3. **Que un fallo en una pregunta no tire la evaluación.** Se provoca uno a
   propósito y se comprueba que las demás siguen.

In [1]:
import sys
import time
from pathlib import Path

RAIZ = Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import json
import subprocess

import pandas as pd

from agente import config, esquema, golden, interfaz

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 60)

RUTA_HOLDOUT = RAIZ / "holdout_simulado.jsonl"
print(config.resumen_configuracion())

{'modelo': 'gpt-5-mini', 'temperatura': 1, 'modelo_embeddings': 'BAAI/bge-small-en-v1.5', 'k': 5, 'tolerancia_cifra': 0.01, 'limite_llamadas_herramienta': 8, 'limite_llamadas_modelo': 10, 'precio_entrada_usd_por_millon': 0.25, 'precio_salida_usd_por_millon': 2.0, 'fecha_precios': '2026-09-21'}


## 1. Las diez preguntas

Se construyen con `agente/golden.py`, el mismo código que levantó el golden
set propio: se escribe la pregunta y un *marcador* —las primeras palabras de
la frase del informe que contiene la respuesta— y el código resuelve el
ancla, sus desplazamientos, el `chunk_id` y las cifras leyéndolos del corpus.
Ni un número ni un desplazamiento se teclean.

**Criterio de selección.** Se han buscado a propósito combinaciones que el
resto de los golden sets no toca:

- Dos preguntas sobre el **Item 7A** (riesgo de mercado), que es la sección
  más corta y la menos representada en los otros conjuntos.
- Ejercicios **FY2024**, que solo aparecen en los otros conjuntos como
  término de comparación y nunca como objeto de la pregunta.
- Conceptos XBRL que no se han consultado antes: flujo de caja de
  explotación, coste de ventas, beneficio por acción diluido.

El reparto por familias imita el del enunciado: cuatro extractivas, tres
numéricas y tres comparativas.

In [2]:
ESPECIFICACION = [
    # --- Extractivas ------------------------------------------------------
    {
        "id": "ho-001", "familia": "extractiva",
        "ticker": "MSFT", "fiscal_year": 2024, "item": "7A",
        "pregunta": "¿Cuáles son las principales exposiciones a divisa que declara "
                    "Microsoft en su 10-K de FY2024?",
        "respuesta_esperada": "El euro, el yen japonés, la libra esterlina, el dólar "
                              "canadiense y el dólar australiano.",
        "marcador": "Principal currency exposures include",
    },
    {
        "id": "ho-002", "familia": "extractiva",
        "ticker": "NVDA", "fiscal_year": 2025, "item": "7A",
        "pregunta": "¿Cómo describe NVIDIA su exposición directa a las fluctuaciones "
                    "de los tipos de cambio en el 10-K de FY2025?",
        "respuesta_esperada": "La considera mínima, porque prácticamente todas sus "
                              "ventas son en dólares y usa contratos a plazo de divisa.",
        "marcador": "We consider our direct exposure to foreign exchange rate",
    },
    {
        "id": "ho-003", "familia": "extractiva",
        "ticker": "AMZN", "fiscal_year": 2025, "item": "1A",
        "pregunta": "¿Qué señala Amazon en los riesgos de su 10-K de FY2025 sobre la "
                    "evolución de la regulación que le afecta?",
        "respuesta_esperada": "Que la regulan muchas jurisdicciones y que el alcance y la "
                              "naturaleza de esa regulación evolucionan y se amplían a "
                              "medida que se amplían sus negocios.",
        "marcador": "A large number of jurisdictions regulate our operations",
    },
    {
        "id": "ho-004", "familia": "extractiva",
        "ticker": "META", "fiscal_year": 2024, "item": "7",
        "pregunta": "¿A qué atribuye Meta el descenso de su tipo impositivo efectivo "
                    "en el ejercicio 2024?",
        "respuesta_esperada": "A los beneficios fiscales en exceso reconocidos por la "
                              "retribución en acciones y a un aumento de las deducciones "
                              "por investigación.",
        "marcador": "Our effective tax rate in 2024 decreased compared to 2023",
    },
    # --- Numéricas --------------------------------------------------------
    {
        "id": "ho-005", "familia": "numerica",
        "ticker": "AAPL", "fiscal_year": 2024, "item": "8",
        "pregunta": "¿Cuánto efectivo generó Apple con sus actividades de explotación "
                    "en el ejercicio fiscal 2024?",
        "concepto": "NetCashProvidedByUsedInOperatingActivities",
    },
    {
        "id": "ho-006", "familia": "numerica",
        "ticker": "GOOGL", "fiscal_year": 2025, "item": "8",
        "pregunta": "¿Cuál fue el coste de los ingresos de Alphabet en 2025?",
        "concepto": "CostOfRevenue",
    },
    {
        "id": "ho-007", "familia": "numerica",
        "ticker": "MSFT", "fiscal_year": 2025, "item": "8",
        "pregunta": "¿Cuál fue el beneficio por acción diluido de Microsoft en el "
                    "ejercicio fiscal 2025?",
        "concepto": "EarningsPerShareDiluted",
    },
    # --- Comparativas -----------------------------------------------------
    {
        "id": "ho-008", "familia": "comparativa",
        "ticker": "MSFT", "fiscal_year": 2025, "fiscal_year_anterior": 2024, "item": "7",
        "pregunta": "¿Cuánto creció el gasto en I+D de Microsoft entre los ejercicios "
                    "fiscales 2024 y 2025, y a qué lo atribuye la dirección?",
        "respuesta_esperada": "Creció 3.000 millones de dólares, un 10 %, por las "
                              "inversiones en ingeniería de nube e IA y en Gaming, "
                              "incluido el efecto de la adquisición de Activision Blizzard.",
        "concepto": "ResearchAndDevelopmentExpense",
        "marcador": "Research and development expenses increased $3.0 billion",
    },
    {
        "id": "ho-009", "familia": "comparativa",
        "ticker": "AAPL", "fiscal_year": 2025, "fiscal_year_anterior": 2024, "item": "7",
        "pregunta": "¿Cómo evolucionó el gasto en I+D de Apple entre 2024 y 2025, y qué "
                    "razones da la compañía?",
        "respuesta_esperada": "Creció, impulsado sobre todo por el aumento de los gastos "
                              "de personal y de los costes de infraestructura.",
        "concepto": "ResearchAndDevelopmentExpense",
        "marcador": "The growth in R&D expense during 2025 compared to 2024",
    },
    {
        "id": "ho-010", "familia": "comparativa",
        "ticker": "AMZN", "fiscal_year": 2025, "fiscal_year_anterior": 2024, "item": "7",
        "pregunta": "¿Cuánto creció el resultado de explotación de Amazon entre 2024 y "
                    "2025, y a qué atribuye la dirección la mejora de AWS?",
        "respuesta_esperada": "El resultado consolidado creció, y la mejora de AWS se "
                              "atribuye al aumento de las ventas, compensado en parte por "
                              "el gasto en infraestructura tecnológica para sostener ese "
                              "crecimiento.",
        "concepto": "OperatingIncomeLoss",
        "marcador": "The increase in AWS operating income in 2025",
    },
]

holdout = [golden.construir({**s, "autor": "holdout-simulado"}) for s in ESPECIFICACION]
print(f"{len(holdout)} preguntas construidas.")

10 preguntas construidas.


## 2. Validación

El mismo validador que el golden set propio, con una excepción: `exigir_20`
va en `False`, porque este conjunto tiene diez preguntas a propósito —son las
diez que dice el enunciado— y no las veinte que exige el entregable.

Se comprueba además, y esto es lo que de verdad importa, que **cada ancla
recorta del texto reconstruido exactamente lo que dice llevar**. Un ancla que
no cuadra convierte una pregunta en imposible sin dar ningún error.

In [3]:
problemas = esquema.validar_golden(holdout, exigir_20=False)
if problemas:
    for p in problemas:
        print(" ", p)
    raise AssertionError(f"{len(problemas)} problemas de validación.")
print("El validador no encuentra problemas.\n")

secciones = {(s["ticker"], s["fiscal_year"], s["item"]): s
             for s in __import__("agente.corpus", fromlist=["x"]).cargar_secciones()}

for item in holdout:
    if not item["ancla_texto"]:
        continue
    texto = secciones[(item["ticker"], item["fiscal_year"], item["item_esperado"])]["texto"]
    recorte = texto[item["ancla_inicio"]:item["ancla_fin"]]
    assert recorte == item["ancla_texto"], item["id"]
    print(f"  {item['id']}  {len(item['ancla_texto'].split()):2d} palabras  "
          f"chunk {item['chunk_id_esperado']}")

print("\nTodas las anclas recortan del texto exactamente lo que declaran.")

El validador no encuentra problemas.

  ho-001  15 palabras  chunk MSFT-2024-7A-0000
  ho-002  39 palabras  chunk NVDA-2025-7A-0001
  ho-003  28 palabras  chunk AMZN-2025-1A-0024
  ho-004  27 palabras  chunk META-2024-7-0027
  ho-008  27 palabras  chunk MSFT-2025-7-0011
  ho-009  21 palabras  chunk AAPL-2025-7-0005
  ho-010  38 palabras  chunk AMZN-2025-7-0019

Todas las anclas recortan del texto exactamente lo que declaran.


In [4]:
# --- Cobertura, para saber qué NO cubre este conjunto ----------------------
resumen = pd.DataFrame(holdout)
print("Por familia:")
print(resumen.familia.value_counts().to_string())
print("\nPor emisor y ejercicio:")
print(pd.crosstab(resumen.ticker, resumen.fiscal_year).to_string())
print("\nPor item:")
print(resumen.item_esperado.value_counts().to_string())

Por familia:
familia
extractiva     4
numerica       3
comparativa    3

Por emisor y ejercicio:
fiscal_year  2024  2025
ticker                 
AAPL            1     1
AMZN            0     2
GOOGL           0     1
META            1     0
MSFT            1     2
NVDA            0     1

Por item:
item_esperado
7     4
8     3
7A    2
1A    1


In [5]:
golden.escribir(RUTA_HOLDOUT, holdout)

# Relectura: un fichero que no se puede volver a leer no está escrito.
releido = [json.loads(l) for l in RUTA_HOLDOUT.open(encoding="utf-8") if l.strip()]
assert len(releido) == 10 and not esquema.validar_golden(releido, exigir_20=False)
print("Relectura correcta.")

holdout_simulado.jsonl: 10 preguntas, 8.1 KB
Relectura correcta.


## 3. El ensayo, cronometrado

Una sola llamada a `evaluar()` con el perfil final, midiendo el tiempo de
pared del proceso entero. Esto es literalmente lo que se va a ejecutar el día
24, con otro fichero.

El presupuesto son veinte minutos para diez preguntas, o sea **dos minutos
por pregunta**. La latencia media del sistema final en el notebook 05 está
muy por debajo, pero la media no es lo que importa aquí: lo que importa es el
peor caso, y el peor caso es una comparativa que agota los topes de llamadas.
Por eso se mira también el máximo, no solo la media.

In [6]:
comienzo = time.perf_counter()
tabla = interfaz.evaluar(RUTA_HOLDOUT, perfil="final", verboso=True)
resumen_holdout = interfaz.resumir(tabla, "final")
segundos = time.perf_counter() - comienzo

tabla.to_csv(config.DIR_RESULTADOS / "eval_holdout_final.csv",
             index=False, encoding="utf-8")

print(f"\n{'=' * 64}")
print(f"TIEMPO DE PARED DE PUNTA A PUNTA: {segundos / 60:.1f} minutos "
      f"({segundos:.0f} s) para {len(tabla)} preguntas")
print(f"Presupuesto del enunciado:        20.0 minutos")
print(f"Margen:                           {(1200 - segundos) / 60:+.1f} minutos")
print(f"Pregunta más lenta:               {tabla.latencia_s.max():.0f} s")
print(f"Coste total:                      ${tabla.coste_usd.sum():.4f}")
print(f"{'=' * 64}")

assert segundos < 1200, (
    "El ensayo no cabe en el presupuesto de veinte minutos. Hay que bajar los "
    "topes de llamadas o el perfil por defecto."
)

Evaluando 10 preguntas · perfil 'final' · modelo gpt-5-mini


  [ 1/10] ho-001     acierto=OK cita=OK cifra=·  tray=OK   46.7s  $0.0035  1 llamadas


  [ 2/10] ho-002     acierto=OK cita=OK cifra=·  tray=OK   25.5s  $0.0055  2 llamadas


  [ 3/10] ho-003     acierto=OK cita=OK cifra=·  tray=OK   23.7s  $0.0068  2 llamadas


  [ 4/10] ho-004     acierto=OK cita=OK cifra=·  tray=OK   23.4s  $0.0056  2 llamadas


...

  [ 6/10] ho-006     acierto=OK cita=·  cifra=OK tray=OK    7.4s  $0.0041  2 llamadas


  [ 7/10] ho-007     acierto=OK cita=·  cifra=OK tray=OK    5.4s  $0.0036  2 llamadas


  [ 8/10] ho-008     acierto=OK cita=OK cifra=OK tray=OK   21.0s  $0.0088  4 llamadas


  [ 9/10] ho-009     acierto=OK cita=OK cifra=OK tray=OK   22.2s  $0.0067  4 llamadas


  [10/10] ho-010     acierto=OK cita=OK cifra=OK tray=OK   25.4s  $0.0089  4 llamadas

  acierto 100.0% · cita 100.0% · cifra 100.0% · trayectoria 100.0%
  coste medio $0.0058/pregunta · total $0.058 · latencia media 21.0 s · 2.5 llamadas/pregunta

TIEMPO DE PARED DE PUNTA A PUNTA: 3.5 minutos (212 s) para 10 preguntas
Presupuesto del enunciado:        20.0 minutos
Margen:                           +16.5 minutos
Pregunta más lenta:               47 s
Coste total:                      $0.0576


In [7]:
print(interfaz.formatear_resumen(resumen_holdout))
print()
print(tabla[["id", "familia", "ticker", "acierto", "cita", "fundamentada",
             "cifra", "trayectoria", "llamadas", "latencia_s"]].to_string(index=False))

  acierto 100.0% · cita 100.0% · cifra 100.0% · trayectoria 100.0%
  coste medio $0.0058/pregunta · total $0.058 · latencia media 21.0 s · 2.5 llamadas/pregunta

    id     familia ticker  acierto cita fundamentada cifra  trayectoria  llamadas  latencia_s
ho-001  extractiva   MSFT     True True         True  None         True         1   46.673061
ho-002  extractiva   NVDA     True True         True  None         True         2   25.506199
ho-003  extractiva   AMZN     True True         True  None         True         2   23.676059
ho-004  extractiva   META     True True         True  None         True         2   23.360640
ho-005    numerica   AAPL     True None         None  True         True         2    9.357819
ho-006    numerica  GOOGL     True None         None  True         True         2    7.429797
ho-007    numerica   MSFT     True None         None  True         True         2    5.364934
ho-008 comparativa   MSFT     True True         True  True         True         4   21

### Lectura del resultado

La comparación que interesa no es con el 100 %, sino con lo que el sistema
saca sobre los conjuntos que sí se han usado para iterar. La diferencia entre
esas dos cifras es la estimación de **cuánto de lo medido en el notebook 05
es ajuste y no procedimiento**.

In [8]:
comparativa = pd.read_csv(config.DIR_RESULTADOS / "tabla_comparativa.csv")
usados = comparativa[(comparativa.sistema == "final")
                     & (comparativa.conjunto.isin(["oficial", "propio"]))]

acierto_usados = float(usados.acierto.mean())
acierto_holdout = float(resumen_holdout["acierto"])

print(f"Acierto sobre los conjuntos usados para iterar: {acierto_usados:.1%}")
print(f"Acierto sobre el hold-out simulado:             {acierto_holdout:.1%}")
print(f"Diferencia:                                     "
      f"{100 * (acierto_holdout - acierto_usados):+.0f} puntos")
print()
print("Con 10 preguntas, una sola vale 10 puntos porcentuales. Una diferencia")
print("menor que eso no se distingue del ruido y no se interpreta.")

Acierto sobre los conjuntos usados para iterar: 77.5%
Acierto sobre el hold-out simulado:             100.0%
Diferencia:                                     +22 puntos

Con 10 preguntas, una sola vale 10 puntos porcentuales. Una diferencia
menor que eso no se distingue del ruido y no se interpreta.


In [9]:
# --- Qué falló, si falló algo ---------------------------------------------
fallos = tabla[tabla.acierto == False]  # noqa: E712
if fallos.empty:
    print("Ninguna pregunta fallida.")
else:
    for _, fila in fallos.iterrows():
        print(f"\n{fila['id']} · {fila['familia']} · {fila['ticker']} "
              f"FY{fila['fiscal_year']}")
        print(f"  pregunta:  {fila['pregunta'][:100]}")
        print(f"  esperado:  {str(fila['respuesta_esperada'])[:100]}")
        print(f"  respondió: {str(fila['respuesta'])[:100]}")
        print(f"  cita {fila['cita']} · fundamentada {fila['fundamentada']} · "
              f"cifra {fila['cifra']} · trayectoria {fila['trayectoria']}")
        print(f"  herramientas: {fila['herramientas']}")

Ninguna pregunta fallida.


## 4. Que funcione sin abrir un notebook

El día 24 nadie va a ejecutar celdas. Se comprueba que la línea de órdenes
hace lo mismo, en un proceso nuevo, con el intérprete y el directorio de
trabajo del repositorio.

Se prueba con dos preguntas y no con las diez para no pagar el ensayo dos
veces: lo que se está comprobando es que el punto de entrada existe y
arranca, no la calidad de las respuestas, que ya está medida arriba.

In [10]:
RUTA_MINI = config.DIR_RESULTADOS / "mini_holdout.jsonl"
golden.escribir(RUTA_MINI, holdout[:2])

orden = [sys.executable, "-m", "agente.interfaz", "evaluar", str(RUTA_MINI)]
print(" ".join(orden), "\n")

proceso = subprocess.run(orden, cwd=RAIZ, capture_output=True, text=True,
                         encoding="utf-8", errors="replace", timeout=900)
print(proceso.stdout[-2500:])
if proceso.returncode != 0:
    print("STDERR:\n", proceso.stderr[-2500:])
assert proceso.returncode == 0, "La línea de órdenes falla."
print("\nLa línea de órdenes funciona en un proceso nuevo.")

mini_holdout.jsonl: 2 preguntas, 1.6 KB
C:\Users\jdmar\AppData\Local\Programs\Python\Python313\python.exe -m agente.interfaz evaluar C:\Users\jdmar\Desktop\Taller NLP\resultados\mini_holdout.jsonl 



Evaluando 2 preguntas · perfil 'final' · modelo gpt-5-mini
  [ 1/2] ho-001     acierto=OK cita=OK cifra=·  tray=OK   43.0s  $0.0038  1 llamadas
  [ 2/2] ho-002     acierto=OK cita=OK cifra=·  tray=OK   19.9s  $0.0056  2 llamadas

Guardado en C:\Users\jdmar\Desktop\Taller NLP\resultados\evaluacion_final.csv y .jsonl

  acierto 100.0% · cita 100.0% · cifra   n/a · trayectoria 100.0%
  coste medio $0.0047/pregunta · total $0.009 · latencia media 31.5 s · 1.5 llamadas/pregunta


La línea de órdenes funciona en un proceso nuevo.


## 5. Que un fallo no tire la evaluación

Con veinte minutos y sin nadie delante, una excepción en la pregunta tres no
puede costar las diecisiete restantes. `evaluar()` captura el error, lo
registra en la fila y sigue.

La prueba usa una pregunta deliberadamente rota —un emisor que no existe en
el corpus— acompañada de dos buenas. Lo que se comprueba es que salen tres
filas y que las dos buenas están bien evaluadas.

In [11]:
ROTA = {
    **holdout[0],
    "id": "ho-rota",
    "ticker": "ZZZZ",
    "pregunta": "¿Qué dice ZZZZ Corporation en su 10-K sobre un riesgo inexistente?",
}

tabla_robustez = interfaz.evaluar([holdout[4], ROTA, holdout[5]],
                                  perfil="baseline", verboso=False)

print(tabla_robustez[["id", "acierto", "error"]].to_string(index=False))
assert len(tabla_robustez) == 3, "Se han perdido filas."
assert tabla_robustez[tabla_robustez.id != "ho-rota"].acierto.notna().all(), (
    "Una pregunta rota ha estropeado la evaluación de las sanas."
)
print("\nLas tres filas están. La pregunta rota no arrastra a las demás.")

     id  acierto error
 ho-005     True  None
ho-rota    False  None
 ho-006     True  None

Las tres filas están. La pregunta rota no arrastra a las demás.


## Qué queda comprobado

| Requisito del §5 | Cómo se ha comprobado |
| --- | --- |
| Diez preguntas nunca vistas | `holdout_simulado.jsonl`, escrito y evaluado una vez |
| En menos de veinte minutos | Tiempo de pared cronometrado, con `assert` sobre el presupuesto |
| Sin abrir un notebook | `python -m agente.interfaz evaluar` en un proceso nuevo |
| Sin que un fallo lo tire todo | Pregunta rota inyectada entre dos sanas |
| Con la clave fuera del código | `api_key.txt`, en `.gitignore`; se comprueba en el notebook 06 |

Lo que **no** queda comprobado, y hay que decirlo: que el sistema acierte
diez preguntas escritas por otra persona. Este conjunto lo hemos escrito
nosotros y, por muy nuevas que sean las preguntas, comparten criterio con las
demás. La única medida libre de ese sesgo en todo el proyecto es el golden
set oficial, y es la que se cita en el informe cuando hace falta una cifra
defendible.